# AI Multi-Agent Research & Fact-Checking System

A multi-agent AI research system that searches the web, evaluates source quality, fact-checks claims, and produces evidence-based answers.

## Key Features

- Live web research
- Multi-source evidence collection
- Source quality evaluation
- Weighted source scoring
- Fact-checking workflow
- Evidence-based synthesis
- Multi-model AI workflow using Groq
- Secure API key handling with Colab Secrets

## System Architecture

```text
User Question
     |
     v
+------------------+
| Research Agent   |
+------------------+
     |
     v
Live Web Search
     |
     v
4 Research Sources
     |
     v
+------------------------+
| Source Evaluator Agent |
+------------------------+
     |
     v
Evidence / Authority / Recency / Relevance
     |
     v
+-----------------------+
| Python Scoring Engine |
+-----------------------+
     |
     v
Source Quality Scores
     |
     v
+------------------+
| Fact Checker     |
+------------------+
     |
     v
Supported / Conflicting / Uncertain Claims
     |
     v
+------------------+
| Synthesis Agent |
+------------------+
     |
     v
Evidence-Based Final Answer
```

## How It Works

1. The **Research Agent** searches the live web for evidence related to the user's question.
2. The system collects multiple sources instead of relying on a single result.
3. The **Source Evaluator Agent** evaluates each source based on evidence quality, authority, recency, and relevance.
4. A deterministic Python scoring engine calculates the final quality score for each source.
5. The **Fact Checker Agent** identifies supported, conflicting, and uncertain claims.
6. The **Synthesis Agent** combines the strongest evidence into a final answer without overstating certainty.

## Source Quality Scoring

### Weights

- Evidence & References: **45%**
- Authority: **30%**
- Recency: **20%**
- Relevance: **5%**

### Quality Levels

- 85–100: **Very Strong Source**
- 70–84: **Strong Source**
- 50–69: **Medium Source**
- Below 50: **Weak Source**

## Agents

- **Research Agent** — searches the web and gathers evidence.
- **Source Evaluator Agent** — evaluates source credibility.
- **Fact Checker Agent** — reviews claims and conflicting evidence.
- **Synthesis Agent** — produces the final evidence-based answer.

## Python Components

- Source quality scoring engine
- Weighted scoring logic
- Source classification
- Structured extraction of evaluation scores

## Tech Stack

- Python
- Groq API
- GPT-OSS / Qwen models
- Browser Search
- Google Colab
- Multi-Agent Architecture
- Fact Checking
- Weighted Source Evaluation

## How to Run

1. Open the notebook in Google Colab.

2. Add your Groq API key to Colab Secrets using the name:

   `GROQ_API_KEY`

3. Enable notebook access for the secret.

4. Run the notebook cells from top to bottom.

5. Enter a research question.

6. The system will search the web, evaluate sources, fact-check claims, and produce a final evidence-based answer.

> Never hard-code or commit API keys to GitHub.

## Example Research Question

`Are electric vehicles better for the environment than gasoline cars?`

## Project Purpose

This project demonstrates how multiple AI agents can collaborate with live web search and deterministic Python scoring to produce more reliable, evidence-based research outputs.

In [ ]:
!pip install -q groq

from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

groq_client = Groq(api_key=GROQ_API_KEY)

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "قول أهلاً بالمصري في جملة قصيرة"
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
def research_agent(question):

    prompt = f"""
    Research this question using current web sources:

    {question}

    Find 4 useful sources.

    For each source, summarize:
    - Title
    - Source type
    - Publication date if available
    - Key evidence
    - Main claim

    Do not assume the conclusion before reviewing the evidence.
    """

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        tools=[
            {
                "type": "browser_search"
            }
        ],
        tool_choice="required"
    )

    return response.choices[0].message.content


research = research_agent(
    "Are electric vehicles better for the environment than gasoline cars?"
)

print(research)

In [ ]:
# =========================================
# SOURCE EVALUATOR AGENT - ROBUST VERSION
# =========================================

import re

def source_evaluator_agent(question, research_text):

    prompt = f"""
You are a source quality evaluator.

Research question:
{question}

Research results:
{research_text}

Evaluate exactly 4 sources.

For every source give scores from 0 to 100 for:
- Evidence quality
- Authority
- Recency
- Relevance

Return EXACTLY 4 lines.

Use this format ONLY:

Source 1 | Evidence: 90 | Authority: 95 | Recency: 80 | Relevance: 100
Source 2 | Evidence: 80 | Authority: 85 | Recency: 90 | Relevance: 95
Source 3 | Evidence: 70 | Authority: 80 | Recency: 75 | Relevance: 90
Source 4 | Evidence: 60 | Authority: 70 | Recency: 85 | Relevance: 80

Do not add explanations.
"""

    response = groq_client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    text = response.choices[0].message.content

    sources = []

    pattern = (
        r"Source\s*(\d+).*?"
        r"Evidence:\s*(\d+).*?"
        r"Authority:\s*(\d+).*?"
        r"Recency:\s*(\d+).*?"
        r"Relevance:\s*(\d+)"
    )

    matches = re.findall(
        pattern,
        text,
        re.IGNORECASE
    )

    for match in matches:

        source_number, evidence, authority, recency, relevance = match

        sources.append({
            "title": f"Source {source_number}",
            "evidence_score": int(evidence),
            "authority_score": int(authority),
            "recency_score": int(recency),
            "relevance_score": int(relevance)
        })

    return sources


question = "Are electric vehicles better for the environment than gasoline cars?"

source_evaluations = source_evaluator_agent(
    question,
    research
)

print(source_evaluations)

In [ ]:
# =========================================
# SOURCE SCORING ENGINE
# =========================================

def calculate_source_quality(source):

    score = (
        source["evidence_score"] * 0.45 +
        source["authority_score"] * 0.30 +
        source["recency_score"] * 0.20 +
        source["relevance_score"] * 0.05
    )

    score = round(score, 2)

    if score >= 85:
        quality = "Very Strong Source"

    elif score >= 70:
        quality = "Strong Source"

    elif score >= 50:
        quality = "Medium Source"

    else:
        quality = "Weak Source"

    return score, quality


scored_sources = []

for source in source_evaluations:

    score, quality = calculate_source_quality(source)

    source["quality_score"] = score
    source["quality"] = quality

    scored_sources.append(source)


for source in scored_sources:
    print(
        source["title"],
        "-",
        source["quality_score"],
        "-",
        source["quality"]
    )

In [ ]:
# =========================================
# FACT CHECKER AGENT
# =========================================

def fact_checker_agent(question, research_text, scored_sources):

    prompt = f"""
You are a strict fact-checking agent.

Question:
{question}

Research:
{research_text}

Source quality scores:
{scored_sources}

Review the evidence.

Identify briefly:

1. Strongly supported claims
2. Conflicting claims
3. Uncertain claims
4. Overall evidence strength

Give more weight to sources with higher quality scores.

Do not invent facts.
"""

    response = groq_client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content


fact_check = fact_checker_agent(
    question,
    research,
    scored_sources
)

print(fact_check)

In [ ]:
# =========================================
# FINAL SYNTHESIS AGENT
# =========================================

def synthesis_agent(
    question,
    research_text,
    scored_sources,
    fact_check
):

    prompt = f"""
You are the final research synthesis agent.

Question:
{question}

Research:
{research_text}

Source quality:
{scored_sources}

Fact-checking report:
{fact_check}

Write a concise final answer.

Rules:
- Base the answer on the evidence.
- Give stronger sources more weight.
- Mention important uncertainty or disagreement.
- Do not exaggerate certainty.
- End with a clear conclusion.
"""

    response = groq_client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content


final_answer = synthesis_agent(
    question,
    research,
    scored_sources,
    fact_check
)

print("=== FINAL RESEARCH ANSWER ===")
print(final_answer)